# CUAD Clause-Labeling Task Replication
## Using Google Gemini API for Extractive Question-Answering

This notebook replicates the primary clause-labeling task from the paper:
**"CUAD: An Expert-Annotated NLP Dataset for Legal Contract Review"**

## Step 1: Install Required Dependencies


In [ ]:
%pip install torch transformers pandas numpy scikit-learn matplotlib seaborn sentencepiece --quiet

Note: you may need to restart the kernel to use updated packages.


## Step 2: Import Required Libraries


In [ ]:
import json
import pandas as pd
import numpy as np
import string
import time
from typing import List, Dict, Any

# Local HF imports (we'll use a local Roberta extractive QA model instead of Gemini)
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering


## Step 3: Configure Gemini API

The client gets the API key from the environment variable `GEMINI_API_KEY`.


In [ ]:
# Configure local model checkpoint (roberta-base) for extractive QA
# If you want to use Gemini later, you can re-introduce genai.configure and client setup.

CKPT_DIR = '../model-checkpoints/roberta-base'  # change if you have another local checkpoint

def load_local_model(ckpt_dir=CKPT_DIR):
    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=False)
    model = AutoModelForQuestionAnswering.from_pretrained(ckpt_dir)
    device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu'))
    try:
        model.to(device)
    except Exception:
        pass
    model.eval()
    return tokenizer, model, device

print('Defined load_local_model()')

AI learns patterns from data to make predictions or decisions.


# Test the API connection with a simple query
try:
    test_response = client.models.generate_content(
        model=model_name, 
        contents="Say 'API test successful' if you can read this."
    )
    print("✓ API test successful!")
    print(f"Model response: {test_response.text}")
except Exception as e:
    print("✗ API test failed!")
    print(f"Error: {str(e)}")
    print("\nPlease check:")
    print("1. Your API key is set in the GEMINI_API_KEY environment variable")
    print("2. You've enabled the Generative Language API")
    print("3. Your API key has the necessary permissions")

## Step 4: Load and Prepare CUAD Dataset

Load the SQuAD-formatted JSON data and create a pandas DataFrame.


In [8]:
# Load CUAD_v1.json file
cuad_json_path = "./datasets/CUAD_v1/CUAD_v1.json"

print("Loading CUAD dataset...")
with open(cuad_json_path, 'r', encoding='utf-8') as f:
    cuad_data = json.load(f)

print(f"✓ CUAD dataset loaded successfully")
print(f"  Number of documents: {len(cuad_data['data'])}")


Loading CUAD dataset...
✓ CUAD dataset loaded successfully
  Number of documents: 510


In [ ]:
# Load CUAD_v1.json and build DataFrame (compact cells)
cuad_json_path = './datasets/CUAD_v1/CUAD_v1.json'
with open(cuad_json_path, 'r', encoding='utf-8') as f:
    cuad_data = json.load(f)

# Parse into a DataFrame
rows = []
for document in cuad_data['data']:
    title = document.get('title', '')
    for paragraph in document.get('paragraphs', []):
        context = paragraph.get('context', '')
        for qa in paragraph.get('qas', []):
            question = qa.get('question', '')
            is_impossible = qa.get('is_impossible', False)
            if is_impossible or len(qa.get('answers', [])) == 0:
                ground_truth = ''
            else:
                ground_truth = qa['answers'][0].get('text', '')
            rows.append({'contract_title': title, 'context': context, 'question': question, 'ground_truth_answer': ground_truth})

import pandas as pd
df_cuad = pd.DataFrame(rows)
print(f'Parsed CUAD: {len(df_cuad)} rows')

In [9]:
# Parse the JSON and create a DataFrame
def parse_cuad_json(cuad_data: Dict[str, Any]) -> pd.DataFrame:
    """
    Parse CUAD JSON data into a pandas DataFrame.
    
    Each row represents a single question-answer pair with columns:
    - contract_title: Title of the contract
    - context: The paragraph/context text
    - question: The question about the clause
    - ground_truth_answer: The answer text (empty string if no answer)
    """
    rows = []
    
    for document in cuad_data['data']:
        contract_title = document['title']
        
        for paragraph in document['paragraphs']:
            context = paragraph['context']
            
            for qa in paragraph['qas']:
                question = qa['question']
                is_impossible = qa.get('is_impossible', False)
                
                # Handle cases with no answer
                if is_impossible or len(qa['answers']) == 0:
                    ground_truth_answer = ""
                else:
                    # Use the first answer's text
                    ground_truth_answer = qa['answers'][0]['text']
                
                rows.append({
                    'contract_title': contract_title,
                    'context': context,
                    'question': question,
                    'ground_truth_answer': ground_truth_answer
                })
    
    return pd.DataFrame(rows)

# Create the DataFrame
print("\nParsing CUAD data into DataFrame...")
df_cuad = parse_cuad_json(cuad_data)

print(f"✓ DataFrame created successfully")
print(f"  Total question-answer pairs: {len(df_cuad)}")
print(f"  Questions with no answer: {(df_cuad['ground_truth_answer'] == '').sum()}")
print(f"  Questions with answer: {(df_cuad['ground_truth_answer'] != '').sum()}")



Parsing CUAD data into DataFrame...
✓ DataFrame created successfully
  Total question-answer pairs: 20910
  Questions with no answer: 14208
  Questions with answer: 6702


In [10]:
# Display the first few rows
print("\nFirst 5 rows of the dataset:")
df_cuad.head()



First 5 rows of the dataset:


,contract_title,context,question,ground_truth_answer
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,DISTRIBUTOR AGREEMENT
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,Distributor
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,"7th day of September, 1999."
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,The term of this Agreement shall be ten (10)...
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,The term of this Agreement shall be ten (10)...


In [11]:
df_cuad.tail()

,contract_title,context,question,ground_truth_answer
20905,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,ENDORSEMENT AGREEMENT entered into by and b...,Highlight the parts (if any) of this contract ...,
20906,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,ENDORSEMENT AGREEMENT entered into by and b...,Highlight the parts (if any) of this contract ...,
20907,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,ENDORSEMENT AGREEMENT entered into by and b...,Highlight the parts (if any) of this contract ...,"Company agrees, at its own expense, to obtain ..."
20908,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,ENDORSEMENT AGREEMENT entered into by and b...,Highlight the parts (if any) of this contract ...,
20909,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,ENDORSEMENT AGREEMENT entered into by and b...,Highlight the parts (if any) of this contract ...,


## Step 5: Define the Clause Extraction Function

This function uses the Gemini API to perform extractive question-answering.


In [ ]:
def extract_clause(contract_context: str, question: str, tokenizer, model, device) -> str:
    """
    Extract a specific clause from contract context using a local extractive QA model.

    Returns the predicted span string, or 'No Answer' if the model predicts a negligible span.
    """
    try:
        inputs = tokenizer(question, contract_context, return_tensors='pt', truncation=True)
        try:
            inputs = {k: v.to(device) for k, v in inputs.items()}
        except Exception:
            pass
        with torch.no_grad():
            out = model(**inputs)
        start_logits = out.start_logits[0].cpu().numpy()
        end_logits = out.end_logits[0].cpu().numpy()
        start = int(start_logits.argmax())
        end = int(end_logits.argmax())
        input_ids = inputs['input_ids'][0].cpu().numpy()
        if end < start or (end - start) > 200:
            return 'No Answer'
        pred_tokens = input_ids[start:(end+1)]
        pred_text = tokenizer.decode(pred_tokens, skip_special_tokens=True).strip()
        if not pred_text:
            return 'No Answer'
        return pred_text
    except Exception as e:
        return f'ERROR: {str(e)}'

print('Replaced extract_clause() with local HF implementation')

✓ extract_clause() function defined


## Step 6: Implement the Evaluation Metric

Calculate Jaccard similarity as described in the CUAD paper.


In [13]:
def calculate_jaccard_similarity(str1: str, str2: str) -> float:
    """
    Calculate Jaccard similarity between two strings.
    
    This replicates the CUAD paper's evaluation metric:
    1. Normalize both strings (lowercase + remove punctuation)
    2. Split into sets of words
    3. Calculate Jaccard: |intersection| / |union|
    
    Args:
        str1: First string (ground truth)
        str2: Second string (prediction)
    
    Returns:
        Jaccard similarity score (float between 0.0 and 1.0)
    """
    # Normalize: lowercase and remove punctuation
    def normalize(text: str) -> str:
        text = text.lower()
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text
    
    # Normalize both strings
    norm_str1 = normalize(str1)
    norm_str2 = normalize(str2)
    
    # Split into sets of words
    set1 = set(norm_str1.split())
    set2 = set(norm_str2.split())
    
    # Handle edge cases
    if len(set1) == 0 and len(set2) == 0:
        # Both are empty (perfect match for "No Answer")
        return 1.0
    
    if len(set1) == 0 or len(set2) == 0:
        # Only one is empty
        return 0.0
    
    # Calculate Jaccard similarity
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    jaccard_score = len(intersection) / len(union)
    
    return jaccard_score

print("✓ calculate_jaccard_similarity() function defined")


✓ calculate_jaccard_similarity() function defined


In [14]:
# Test the Jaccard similarity function
print("\nTesting Jaccard similarity function:")
print(f"Same text: {calculate_jaccard_similarity('Hello world', 'Hello world'):.3f}")
print(f"Different text: {calculate_jaccard_similarity('Hello world', 'Goodbye moon'):.3f}")
print(f"Partial overlap: {calculate_jaccard_similarity('Hello world test', 'Hello test'):.3f}")
print(f"Both empty: {calculate_jaccard_similarity('', ''):.3f}")
print(f"One empty: {calculate_jaccard_similarity('Hello', ''):.3f}")



Testing Jaccard similarity function:
Same text: 1.000
Different text: 0.000
Partial overlap: 0.667
Both empty: 1.000
One empty: 0.000


In [ ]:
# Implement Jaccard similarity (as requested)
import string

def calculate_jaccard_similarity(str1: str, str2: str) -> float:
    def normalize(text: str) -> str:
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text
    norm1 = normalize(str1)
    norm2 = normalize(str2)
    set1 = set(norm1.split())
    set2 = set(norm2.split())
    if len(set1) == 0 and len(set2) == 0:
        return 1.0
    if len(set1) == 0 or len(set2) == 0:
        return 0.0
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    return len(intersection) / len(union)

print('Defined calculate_jaccard_similarity()')

## Step 7: Run the Experiment on a Sample

Test the implementation on the first 10 rows of the dataset.


In [15]:
# Create a sample of the first 10 rows
sample_size = 10
df_sample = df_cuad.head(sample_size).copy()

print(f"Created sample DataFrame with {len(df_sample)} rows")
print(f"\nSample distribution:")
print(f"  Questions with answer: {(df_sample['ground_truth_answer'] != '').sum()}")
print(f"  Questions with no answer: {(df_sample['ground_truth_answer'] == '').sum()}")


Created sample DataFrame with 10 rows

Sample distribution:
  Questions with answer: 7
  Questions with no answer: 3


In [ ]:
# Run the experiment on a small sample (10 rows) using local model
# Load model
tokenizer, model, device = load_local_model()

# Prepare sample
sample_size = 10
df_sample = df_cuad.head(sample_size).copy()

predicted_answers = []
jaccard_scores = []
for idx, row in df_sample.iterrows():
    pred = extract_clause(row['context'], row['question'], tokenizer, model, device)
    score = calculate_jaccard_similarity(row['ground_truth_answer'], pred)
    predicted_answers.append(pred)
    jaccard_scores.append(score)
    print(f"{idx+1}/{sample_size} jaccard={score:.3f}")

# Attach results
df_sample['predicted_answer'] = predicted_answers
df_sample['jaccard_score'] = jaccard_scores

print('Done. Average Jaccard:', df_sample['jaccard_score'].mean())

df_sample.head()

In [16]:
# Run the experiment
print("\n" + "="*80)
print("RUNNING EXPERIMENT ON SAMPLE DATA")
print("="*80)
print(f"\nProcessing {len(df_sample)} questions...\n")

predicted_answers = []
jaccard_scores = []

for idx, row in df_sample.iterrows():
    print(f"Processing question {idx + 1}/{len(df_sample)}...")
    
    # Extract clause using Gemini
    predicted_answer = extract_clause(row['context'], row['question'])
    predicted_answers.append(predicted_answer)
    
    # Calculate Jaccard similarity
    jaccard_score = calculate_jaccard_similarity(row['ground_truth_answer'], predicted_answer)
    jaccard_scores.append(jaccard_score)
    
    print(f"  Jaccard Score: {jaccard_score:.3f}")
    print()

# Add results to DataFrame
df_sample['predicted_answer'] = predicted_answers
df_sample['jaccard_score'] = jaccard_scores

print("✓ Experiment completed!")



RUNNING EXPERIMENT ON SAMPLE DATA

Processing 10 questions...

Processing question 1/10...
  Jaccard Score: 0.500

Processing question 2/10...
  Jaccard Score: 0.043

Processing question 3/10...
  Jaccard Score: 0.217

Processing question 4/10...
  Jaccard Score: 1.000

Processing question 5/10...
  Jaccard Score: 1.000

Processing question 6/10...
  Jaccard Score: 0.723

Processing question 7/10...
  Jaccard Score: 0.000

Processing question 8/10...
  Jaccard Score: 1.000

Processing question 9/10...
  Jaccard Score: 0.000

Processing question 10/10...
  Jaccard Score: 0.000

✓ Experiment completed!


## Step 8: Display Results

View the predictions and evaluation metrics.


In [17]:
# Display the complete results
print("\n" + "="*80)
print("EXPERIMENT RESULTS")
print("="*80 + "\n")

# Show all columns with better formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

df_sample



EXPERIMENT RESULTS



,contract_title,context,question,ground_truth_answer,predicted_answer,jaccard_score
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Document Name"" that should be reviewed...",DISTRIBUTOR AGREEMENT,"THIS DISTRIBUTOR AGREEMENT (the ""Agreement"")",0.500000
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Parties"" that should be reviewed by a ...",Distributor,"THIS DISTRIBUTOR AGREEMENT (the ""Agreement"") is made by and between Electric City Corp., a ...",0.043478
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Agreement Date"" that should be reviewe...","7th day of September, 1999.","THIS DISTRIBUTOR AGREEMENT (the ""Agreement"") is made by and between Electric City Corp., a ...",0.217391
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Effective Date"" that should be reviewe...","The term of this Agreement shall be ten (10) years (the ""Term"") wh...","The term of this Agreement shall be ten (10)\nyears (the ""Term"") which shall commence on the...",1.000000
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Expiration Date"" that should be review...","The term of this Agreement shall be ten (10) years (the ""Term"") wh...","The term of this Agreement shall be ten (10)\nyears (the ""Term"") which shall commence on the...",1.000000
5,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Renewal Term"" that should be reviewed ...","If Distributor complies with all of the terms of this Agreement, the ...","The term of this Agreement shall be ten (10)\nyears (the ""Term"") which shall commence on the...",0.723404
6,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Notice Period To Terminate Renewal"" th...",,No Answer,0.000000
7,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Governing Law"" that should be reviewed...",This Agreement is to be construed according to the laws of the State of Illinois.,This Agreement is to be construed according to the laws\n of the State of Illinois.,1.000000
8,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Most Favored Nation"" that should be re...",,No Answer,0.000000
9,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n DISTRIBUTOR AGREEMENT\n\n THIS DISTRIBUTO...,"Highlight the parts (if any) of this contract related to ""Non-Compete"" that should be reviewed b...",,(B) Distributor further agrees that it will not interfere\n with ...,0.000000


In [18]:
# Calculate and display the average Jaccard score
average_jaccard = df_sample['jaccard_score'].mean()

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"\nAverage Jaccard Similarity: {average_jaccard:.4f}")
print(f"Median Jaccard Similarity: {df_sample['jaccard_score'].median():.4f}")
print(f"Min Jaccard Similarity: {df_sample['jaccard_score'].min():.4f}")
print(f"Max Jaccard Similarity: {df_sample['jaccard_score'].max():.4f}")
print(f"Std Dev: {df_sample['jaccard_score'].std():.4f}")

# Additional metrics
print(f"\nPerfect matches (score = 1.0): {(df_sample['jaccard_score'] == 1.0).sum()}")
print(f"High scores (score >= 0.7): {(df_sample['jaccard_score'] >= 0.7).sum()}")
print(f"Medium scores (0.3 <= score < 0.7): {((df_sample['jaccard_score'] >= 0.3) & (df_sample['jaccard_score'] < 0.7)).sum()}")
print(f"Low scores (score < 0.3): {(df_sample['jaccard_score'] < 0.3).sum()}")



SUMMARY STATISTICS

Average Jaccard Similarity: 0.4484
Median Jaccard Similarity: 0.3587
Min Jaccard Similarity: 0.0000
Max Jaccard Similarity: 1.0000
Std Dev: 0.4487

Perfect matches (score = 1.0): 3
High scores (score >= 0.7): 4
Medium scores (0.3 <= score < 0.7): 1
Low scores (score < 0.3): 5


## Step 9: Detailed Results Inspection

Examine individual predictions in detail.


In [19]:
# Display detailed results for each question
print("\n" + "="*80)
print("DETAILED RESULTS FOR EACH QUESTION")
print("="*80 + "\n")

for idx, row in df_sample.iterrows():
    print(f"Question {idx + 1}:")
    print("-" * 80)
    print(f"Contract: {row['contract_title'][:80]}...")
    print(f"\nQuestion: {row['question']}")
    print(f"\nGround Truth Answer:")
    if row['ground_truth_answer'] == "":
        print("  [No Answer]")
    else:
        print(f"  {row['ground_truth_answer'][:200]}..." if len(row['ground_truth_answer']) > 200 else f"  {row['ground_truth_answer']}")
    
    print(f"\nPredicted Answer:")
    if row['predicted_answer'] == "":
        print("  [No Answer]")
    else:
        print(f"  {row['predicted_answer'][:200]}..." if len(row['predicted_answer']) > 200 else f"  {row['predicted_answer']}")
    
    print(f"\nJaccard Score: {row['jaccard_score']:.4f}")
    print("\n" + "="*80 + "\n")



DETAILED RESULTS FOR EACH QUESTION

Question 1:
--------------------------------------------------------------------------------
Contract: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT...

Question: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract

Ground Truth Answer:
  DISTRIBUTOR AGREEMENT

Predicted Answer:
  THIS DISTRIBUTOR AGREEMENT (the "Agreement")

Jaccard Score: 0.5000


Question 2:
--------------------------------------------------------------------------------
Contract: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT...

Question: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract

Ground Truth Answer:
  Distributor

Predicted Answer:
  THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Compa

## Step 10: Export Results (Optional)

Save the results to a CSV file for further analysis.


In [ ]:
# Export results to CSV
output_path = "./generated_files/cuad_gemini_results_sample.csv"
df_sample.to_csv(output_path, index=False)

print(f"✓ Results exported to: {output_path}")



To run the full experiment on the entire CUAD dataset:

1. Replace `sample_size = 10` with the desired number of samples (or use `df_cuad` instead of `df_sample`)
2. API rate limits (free tier: 60 queries/minute)
3. Error handling and retry logic for large-scale experiments
4. Monitor API costs if using a paid tier

### Comparing to CUAD Paper Results

The CUAD paper reports the following baseline results:
- BERT-base: 0.237 (AUPR)
- RoBERTa-base: 0.320 (AUPR)
- DeBERTa-large: 0.365 (AUPR)

Note: The paper uses Area Under Precision-Recall (AUPR) as the primary metric, but Jaccard similarity provides an intuitive measure of text overlap.
